In [5]:
!nvidia-smi
!git clone https://github.com/KUO-WEL/CA_Final_Project.git
%cd CA_Final_Project

Tue Jun  2 07:36:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
%%writefile main.cu
#include <iostream>
#include <cuda_runtime.h>
#include "include/dataset.h" // 引入你的測資

// 定義我們要一次平行處理的封包數量 (用來填滿 GPU)
#define NUM_PACKETS 64

// ========================================================
// CUDA Kernel: 2D Grid 平行化架構
// ========================================================
__global__ void cross_correlation_2d_kernel(const float* d_rx_all, const float* d_preamble, float* d_result_all, int num_windows) {
    // 1. 取得 2D Grid 的索引
    int window_idx = blockIdx.x * blockDim.x + threadIdx.x; // X軸：負責封包內的滑動視窗
    int packet_idx = blockIdx.y;                            // Y軸：負責區分現在是第幾個封包

    // 2. 宣告 Shared Memory 用來存放 Preamble (每個 Block 獨立擁有一份)
    __shared__ float s_preamble[PREAMBLE_LEN];

    // 3. 協同載入 (Collaborative Loading)
    int local_tid = threadIdx.x;
    if (local_tid < PREAMBLE_LEN) {
        s_preamble[local_tid] = d_preamble[local_tid];
    }
    __syncthreads(); // 等待 Preamble 載入完成

    // 4. 執行相關性計算
    if (window_idx < num_windows) {
        // [關鍵改變] 根據 packet_idx 計算這個封包專屬的記憶體「起點偏移量」
        const float* current_rx = d_rx_all + (packet_idx * RX_LEN);
        float* current_result = d_result_all + (packet_idx * num_windows);

        float sum = 0.0f;
        for (int j = 0; j < PREAMBLE_LEN; ++j) {
            sum += s_preamble[j] * current_rx[window_idx + j];
        }

        // 將結果寫入這個封包專屬的位置
        current_result[window_idx] = sum;
    }
}

// ========================================================
// Host 主程式
// ========================================================
int main() {
    int num_windows = RX_LEN - PREAMBLE_LEN + 1;

    // 計算 64 個封包需要的總記憶體大小
    size_t rx_bytes = NUM_PACKETS * RX_LEN * sizeof(float);
    size_t preamble_bytes = PREAMBLE_LEN * sizeof(float);
    size_t result_bytes = NUM_PACKETS * num_windows * sizeof(float);

    // 1. 準備 Host 端的巨量資料 (擴充測資)
    float* h_rx_all = new float[NUM_PACKETS * RX_LEN];
    float* h_result_all = new float[NUM_PACKETS * num_windows];

    // 將 dataset.h 中的 8 個 Pattern 循環填入 64 個封包的空間中
    for(int i = 0; i < NUM_PACKETS; ++i) {
        int pattern_id = i % 8; // 0~7 循環
        memcpy(&h_rx_all[i * RX_LEN], &RX_SIGNALS[pattern_id * RX_LEN], RX_LEN * sizeof(float));
    }

    // 2. 配置 GPU (Device) 記憶體
    float *d_rx_all, *d_preamble, *d_result_all;
    cudaMalloc((void**)&d_rx_all, rx_bytes);
    cudaMalloc((void**)&d_preamble, preamble_bytes);
    cudaMalloc((void**)&d_result_all, result_bytes);

    // 3. 將資料從 CPU 拷貝到 GPU (一次送出 64 個封包)
    cudaMemcpy(d_rx_all, h_rx_all, rx_bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_preamble, PREAMBLE, preamble_bytes, cudaMemcpyHostToDevice);

    // 4. [最核心的改變] 設定 2D Grid
    int threads_per_block = 256;
    int blocks_per_grid_x = (num_windows + threads_per_block - 1) / threads_per_block; // X軸: 16

    // 定義 2D Grid: (X=16 個視窗區塊, Y=64 個封包) -> 總共啟動 1024 個 Blocks!
    dim3 block(threads_per_block);
    dim3 grid(blocks_per_grid_x, NUM_PACKETS);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // 5. 啟動 Kernel
    cudaEventRecord(start);
    cross_correlation_2d_kernel<<<grid, block>>>(d_rx_all, d_preamble, d_result_all, num_windows);
    cudaEventRecord(stop);

    // 6. 將運算結果拷貝回 CPU
    cudaMemcpy(h_result_all, d_result_all, result_bytes, cudaMemcpyDeviceToHost);
    cudaEventSynchronize(stop);

    float ms;
    cudaEventElapsedTime(&ms, start, stop);

    // 7. 驗證 64 個封包的結果是否全部正確
    std::cout << "=== Part 5: GPU 2D Grid Parallelism ===" << std::endl;
    std::cout << "同時處理封包數: " << NUM_PACKETS << std::endl;
    std::cout << "總啟動 Blocks 數量: " << grid.x * grid.y << std::endl;
    std::cout << "Kernel 執行時間: " << ms << " ms" << std::endl;

    int error_count = 0;
    for (int p = 0; p < NUM_PACKETS; ++p) {
        float max_corr = -1e9;
        int detected_idx = -1;
        float* current_result = &h_result_all[p * num_windows];

        for (int i = 0; i < num_windows; ++i) {
            if (current_result[i] > max_corr) {
                max_corr = current_result[i];
                detected_idx = i;
            }
        }

        // 驗證是否與正確解答相符
        if (detected_idx != GROUND_TRUTH[p % 8]) {
            error_count++;
        }
    }

    if (error_count == 0) {
        std::cout << "=> GPU 2D Grid 驗證成功！64 個封包全部命中！" << std::endl;
    } else {
        std::cout << "=> 驗證失敗！有 " << error_count << " 個封包計算錯誤。" << std::endl;
    }

    // 8. 釋放記憶體
    cudaFree(d_rx_all);
    cudaFree(d_preamble);
    cudaFree(d_result_all);
    delete[] h_rx_all;
    delete[] h_result_all;

    return 0;
}

Writing main.cu


In [7]:
!nvcc -Xptxas -v main.cu -o main -O2 -arch=sm_75
!./main

ptxas info    : 0 bytes gmem
ptxas info    : Compiling entry function '_Z27cross_correlation_2d_kernelPKfS0_Pfi' for 'sm_75'
ptxas info    : Function properties for _Z27cross_correlation_2d_kernelPKfS0_Pfi
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 67 registers, used 1 barriers, 1024 bytes smem, 380 bytes cmem[0]
ptxas info    : Compile time = 47.768 ms
=== Part 5: GPU 2D Grid Parallelism ===
同時處理封包數: 64
總啟動 Blocks 數量: 1024
Kernel 執行時間: 0.330016 ms
=> GPU 2D Grid 驗證成功！64 個封包全部命中！


In [8]:
!ncu --set basic ./main

==PROF== Connected to process 2995 (/content/CA_Final_Project/CA_Final_Project/main)
==PROF== Profiling "cross_correlation_2d_kernel" - 0: 0%....50%....100% - 9 passes
=== Part 5: GPU 2D Grid Parallelism ===
同時處理封包數: 64
總啟動 Blocks 數量: 1024
Kernel 執行時間: 2066.13 ms
=> GPU 2D Grid 驗證成功！64 個封包全部命中！
==PROF== Disconnected from process 2995
[2995] main@127.0.0.1
  cross_correlation_2d_kernel(const float *, const float *, float *, int) (16, 64, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.96
    SM Frequency                    Mhz       584.87
    Elapsed Cycles                cycle      172,733
    Memory Throughput                 %        72.56
    DRAM Throughput                   %         2.84
    Duration                         u